# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import dataset
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os

train_df, val_df, test_df = dataset.load_split(LABELS_DIR, 0)
stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)

plt.figure(figsize=(10,4))
sns.countplot(data=train_df, x='Label')
plt.title('Train Label Distribution')
plt.show()

for label in range(9):
    subset = train_df[train_df['Label'] == label].head(3)
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    for i, (_, row) in enumerate(subset.iterrows()):
        img_path = os.path.join(IMAGES_DIR, row['Filename'])
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[i].imshow(img)
            axes[i].set_title(f'Class {label}')
            axes[i].axis('off')
    plt.show()


In [ ]:
import train, model, losses
import torch
train.set_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dummy_model = model.build_model('resnet50', pretrained=True, num_classes=9).to(device)
criterion = losses.build_criterion('ce')
optimizer = torch.optim.Adam(dummy_model.parameters(), lr=1e-3)

train_tf = dataset.build_transforms(True, 224, 'basic')
tiny_loader = dataset.make_loader(train_df.head(8), IMAGES_DIR, train_tf, 8, True, None, 0)
images, targets, _ = next(iter(tiny_loader))
images, targets = images.to(device), targets.to(device)

initial_logits = dummy_model(images)
print('Initial Loss:', criterion(initial_logits, targets).item())

for i in range(20):
    optimizer.zero_grad()
    logits = dummy_model(images)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()
print('Final Overfit Loss:', criterion(dummy_model(images), targets).item())


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from train import Config, run
import pandas as pd
results_b = []
backbones = ['resnet50', 'efficientnet_b0', 'mobilenetv3']
for bb in backbones:
    cfg = Config(exp_id=f'B_{bb}', backbone=bb, seed=0, epochs=2)
    res = run(cfg)
    res['backbone'] = bb
    results_b.append(res)

df_b = pd.DataFrame(results_b)
display(df_b)


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
from train import Config, run
results_t = []
cfg_t00 = Config(exp_id='T00', backbone='resnet50', seed=0, epochs=3)
results_t.append(run(cfg_t00))

cfg_t01 = Config(exp_id='T01', backbone='resnet50', seed=0, epochs=3, loss='focal')
results_t.append(run(cfg_t01))

cfg_t02 = Config(exp_id='T02', backbone='resnet50', seed=0, epochs=3, mix='mixup')
results_t.append(run(cfg_t02))

df_t = pd.DataFrame(results_t)
display(df_t)


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# Code inference tuỳ chọn dựa trên inference.py
print('Thực hiện Inference... (cần load best model và benchmark)')


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
for seed in (0, 1, 2):
    run(Config(exp_id='T00', backbone='resnet50', seed=seed, epochs=12, save_test_predictions=True))
    run(Config(exp_id='F01', backbone='resnet50', loss='focal', mix='mixup', seed=seed, epochs=12, save_test_predictions=True))


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
import pandas as pd
with pd.ExcelWriter('results.xlsx', engine='openpyxl') as writer:
    df_b.to_excel(writer, sheet_name='Backbones', index=False)
    df_t.to_excel(writer, sheet_name='Training', index=False)
print('Lưu kết quả thành công!')
